# PRÁCTICA PR3 (PARTE GUIADA) | INFERENCIA PRyES
## Trabajo con temperaturas mínimas y máximas de España (fuente: AEMET)
Objetivos de aprendizaje
- Caracterizar la distribución de la media muestral a partir de datos reales
- Calcular el error estándar del estimador media muestral de forma teórica
- Estimar un interalo de confianza para la media muestral a partir de los datos reales
- Utilizar Bootstrap para estimar el error estandar de un estimador
- Utilizar Bootstrap para estimar un intervalo de confianza para una variable poblacional
- Estimar por máxima verosimilitud un conjunto de parámetros a partir de una muestra


### 0: Carga de librerías necesarias

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
from scipy import stats
from scipy import optimize as opt  # para estimación por máxima verosimilitud
import matplotlib.pyplot as plt
import seaborn as sns

### 1: Carga de datos

In [ ]:
# leemos los datos del fichero con las temperaturas usando pandas
# Los datos están en ºC

# df = pd.read_csv('TemperaturesSpain.csv', index_col=['YEAR','MONTH','DATE'])
df_orig = pd.read_csv('TemperaturesSpain.csv')

# parseamos la fecha (cadena) para que sea un datetime con formato dd/mm/yyyy
df_orig["DATE"] = pd.to_datetime(df_orig["DATE"], format='%d/%m/%Y') 
print('Tamaño de df_orig con los datos cargados:', df_orig.shape)
print('Dataframe con todos los datos leidos (filas: días, columnas: variables temperaturas):')
df_orig

# 1. Estimación de la media de TMAX238 (Zaragoza Aeropuerto) utilizando enfoque teórico

### 1.1. ¿Cuál es el valor medio de TMAX238?

Usar el estimador media muestral

In [ ]:
x = df_orig.TMAX238

n = len(x)
print(f'Muestra de tamaño n={n}, bastante grande (datos desde {df_orig.DATE[0]} a {df_orig.DATE[n-1]})')

# usando el estimador media muestral
x_media = np.mean(x)
print(f'Estimación del valor medio de TMAX238 usando la media muestral a partir de la muestra de tamaño {n} = {x_media} ºC')

# representamos datos de la muestra y el valor de la estimación puntual de la media poblacional
plt.figure(1, figsize=(15, 9))
plt.subplot(2,1,1)
plt.plot(x, label=f'Muestra tamaño {n} (ordenada temporalmente)')
plt.hlines(x_media, 0, len(x), color='black', label=f'Estimación media TMAX238 = {x_media}')
plt.ylabel('TMAX238')
plt.legend()

plt.subplot(2,1,2)
plt.plot(np.random.permutation(x), '.', label=f'Muestra desordenada tamaño {n}')
plt.plot([0, len(x)], [x_media, x_media], 'black', label=f'Estimación media TMAX238 = {x_media}')
plt.ylabel('TMAX238')
plt.title('Datos desordenados')
plt.legend()
plt.show()


### 1.2. ¿TMAX238 se distribuye como una normal?


In [ ]:
# Distribución empírica de la variable bajo estudio

plt.figure(1, figsize=(15, 6))
plt.hist(x, 40, label=f'Histograma Muestra tamaño {n}')
plt.vlines(x_media, 0, 500, color='black', label=f'Estimación media TMAX238 = {x_media}')
plt.xlabel('TMAX238')
plt.legend()
plt.show()

print('La variable TMAX238 no parece distribuirse según una normal.')

### 1.3. ¿Cuál es la distribución del estimador utilizado para determinar el valor medio de TMAX238?

Recuerda: $\bar{X}\sim{\mathcal{N}(\mu,\sigma^2/n)}$ si $X\sim{\mathcal{N}(\mu,\sigma^2)}$   y  $\sigma^2$ es conocida. Pero si la $\sigma^2$ es desconocida, se utiliza una distribución t de Student. Además, si n es grande, entonces dicha t se puede aproximar por una normal ...

In [ ]:
# Distribución de la media muestral
print('La media muestral con varianza poblacional desconocida sigue una distribución t.')
print('Pero como n es grande, seguro que se puede aproximar bastante bien por una normal:')

# calculamos la varianza muestral sx
sx = np.std(x, ddof=1) # <--------- indicando 1 significa que se divide por n-1
print(f' -  Varianza muestral calculada = {sx**2}')

# Ahora creamos la v.a. MEDIA MUESTRAL
#    De la ayuda de scipy.stats.t :
#    To shift and/or scale the distribution use the loc and scale parameters. 
#    Specifically, t.pdf(x, df, loc, scale) is identically equivalent to t.pdf(y, df) / scale with y = (x - loc) / scale.

Xmedia_t = stats.t(n-1, loc=np.mean(x), scale = sx/np.sqrt(n))  # <----------------------- ojo 
print(f' -  Media de la v.a. Xmedia_t (t-student) = {Xmedia_t.mean()} y varianza = {Xmedia_t.var()}')

Xmedia_N = stats.norm(loc=np.mean(x), scale = sx/np.sqrt(n))
print(f' -  Media de la v.a. Xmedia_N (Normal)    = {Xmedia_N.mean()} y varianza = {Xmedia_N.var()}')

# puntos para representar
xs = np.linspace(Xmedia_t.ppf(0.000001), Xmedia_t.ppf(1-0.000001), 1000)

plt.figure(1, figsize=(15, 6))
ax1 = plt.subplot(2,1,1)
plt.plot(xs, Xmedia_t.pdf(xs), label=f'Distribución Media muestral (según una t)')
plt.plot(xs, Xmedia_N.pdf(xs), label=f'Distribución Media muestral (según una Normal)')
plt.vlines(x_media, 0, np.max(Xmedia_t.pdf(xs)), color='black', label=f'Estimación media TMAX238')
plt.title('Distribución del estimador media muestral de TMAX238')
plt.legend()
plt.subplot(2,1,2, sharex = ax1)
plt.stem(xs, Xmedia_t.pdf(xs) -  Xmedia_N.pdf(xs),markerfmt='none')
plt.title('Diferencia entre pdfs')
plt.show()

print('Según las diferencias entre las pdfs, la aproximación de la t por una normal es muy razonabla') 

### 1.4. Calcular el error estándar del estimador media muestral para TMAX238. ¿Cuál es la probabilidad de que la media de TMAX238 sea mayor que 22 ºC?

In [ ]:
# Error estándar de la media muestral
se_Xmedia = sx/np.sqrt(n)
print( f'El error estándar del estimador media muestral es la desviación típica de dicho estimador, vale = {se_Xmedia} ºC')

# cálculo de la probabilidad a partir de la distribución de la media muestral
prob = 1 - Xmedia_t.cdf(22)
print('La probabilidad de que la media de TMAX238 sea mayor que 22 se puede estimar a partir de la distribución del estimador Xmedia.')
print(f'Dicha probabilidad vale : 1 - P(Xmedia_t<22)={prob}')

### 1.5. Calcular, sin usar bootstrap, un intervalo de confianza para la media de TMAX238, con un nivel de confianza del 90%.

In [ ]:
print('Para calcular el IC_90%(mu) usamos la distribución teórica de la media muestral (t de Student)')

gama = 0.90
alfa = 1 - gama
lim_inf = Xmedia_t.ppf(alfa/2)
lim_sup = Xmedia_t.ppf(1-alfa/2)

print(f'El intervalo al {100*gama}% de la media de TMAX238 es: [{lim_inf}, {lim_sup}]')

# Gráficamente

# puntos para representar la distribución y el intervalo
xs = np.linspace(Xmedia_t.ppf(0.00001), Xmedia_t.ppf(1-0.00001), 1000)
xIC = np.linspace(lim_inf, lim_sup, 1000)

plt.figure(1, figsize=(15, 6))
plt.plot(xs, Xmedia_t.pdf(xs), linewidth = 4, label=f'Distribución Media muestral (según una t)')
plt.fill_between( xIC, Xmedia_t.pdf(xIC), color="green", alpha=0.3, label=f'Probabilidad asociada ($\gamma=${gama})')
plt.vlines(x_media, 0, np.max(Xmedia_t.pdf(xs)), color='black', label=f'Estimación puntual media TMAX238')
plt.errorbar(x_media, -0.1, xerr = np.array([[x_media - lim_inf], [lim_sup - x_media]]), fmt = 'o', label = f'IC [{lim_inf}, {lim_sup}] al {gama*100}%')
plt.title('Distribución del estimador media muestral de TMAX238 e IC_90%')
plt.tight_layout()
plt.legend()
plt.show()

# representación sobre la serie temporal
plt.figure(2, figsize=(15, 5))
plt.plot(x, label=f'Muestra tamaño {n} (ordenada temporalmente)')
plt.hlines(x_media, 0, len(x), color='black', label=f'Estimación puntual media TMAX238 = {x_media}')
plt.fill_between([0, len(x)], lim_inf, lim_sup, color="green", alpha=0.3, label = f'IC media [{lim_inf}, {lim_sup}] al {gama*100}%')
plt.ylim([14, 29])
plt.tight_layout()
plt.legend()
plt.show()

# 2. Estimación por intervalo de la media de TMAX238 utilizando bootstrap

### 2.1 Estima el IC(media) utilizando el método "normal"

In [ ]:
# calculamos el error estándar del estimador con bootstrap

muestra_orig = (x,)  # tiene que ser una secuencia

# params para llamar a bootstrap
R = 9999
estadistico = np.mean
nivel_confianza = 0.90
alpha = 1- nivel_confianza

# Método Normal (basado en el SE estimado con bootstrap)
resNor = stats.bootstrap(muestra_orig, estadistico, confidence_level=nivel_confianza,  n_resamples=R, random_state= 999)
SE = resNor.standard_error
estimate_full_sample = np.mean(muestra_orig)
va = stats.t(n-1)
valor_critico = va.ppf(1-alpha/2)
lim_inf_Nor = estimate_full_sample - valor_critico * SE
lim_sup_Nor = estimate_full_sample + valor_critico * SE

print(f'Valor crítico: {valor_critico}')
print(f'IC {100*nivel_confianza}% estimado según método Normal: [{lim_inf_Nor}, {lim_sup_Nor}]')

### 2.2 Estima el IC(media) utilizando el método "BCa"

In [ ]:
# Método BCa
muestra_orig = (x,)  # tiene que ser una secuencia

# params para llamar a bootstrap
R = 9999
estadistico = np.mean
nivel_confianza = 0.90
alpha = 1- nivel_confianza

res_BCa = stats.bootstrap(muestra_orig, estadistico, confidence_level=nivel_confianza,  n_resamples=R, method = 'BCA', random_state= 999)
# devuelve como resultado el IC y el SE del estadístico
print(res_BCa)
lim_inf_BCa, lim_sup_BCa = res_BCa.confidence_interval

print(f'IC {100*nivel_confianza}% estimado según método BCa: [{lim_inf_BCa}, {lim_sup_BCa}]')

### 2.3 Compara gráficamente los IC obtenidos con Bootstrap con el tradicional

In [ ]:
# prepara datos para representar
centro_int_conf = x_media
lims_int_conf_trad  = np.array([[centro_int_conf - lim_inf],     [lim_sup     - centro_int_conf]])
lims_int_conf_Nor   = np.array([[centro_int_conf - lim_inf_Nor], [lim_sup_Nor - centro_int_conf]])
lims_int_conf_BCa   = np.array([[centro_int_conf - lim_inf_BCa], [lim_sup_BCa - centro_int_conf]])

plt.figure(1, figsize=(15, 2))
plt.errorbar(centro_int_conf, 0, xerr = lims_int_conf_trad, fmt = 'o', 
             label = f'IC [{lim_inf}, {lim_sup}] al {gama*100}%')
plt.errorbar(centro_int_conf, 1, xerr = lims_int_conf_Nor, fmt = 'o', 
             label = f'IC [{lim_inf_Nor}, {lim_sup_Nor}] al {gama*100}%')
plt.errorbar(centro_int_conf, 2, xerr = lims_int_conf_BCa, fmt = 'o', 
             label = f'IC [{lim_inf_BCa}, {lim_sup_BCa}] al {gama*100}%')
plt.yticks([0, 1, 2, 5],['IC tradicional', 'IC bootstrap Normal', 'IC bootstrap BCa', ''])
plt.legend()
plt.show()

# 3. Estimación por máxima verosimilitud de TMAX238 a partir de TMIN238

### 3.1 Analizar la relación entre TMAX238 y TMIN238

In [ ]:
# definimos variables por comodidad
x = df_orig.TMIN238
y = df_orig.TMAX238

In [ ]:
# representación de la muestra de datos

plt.figure(1, figsize=(15, 5))
ax1=plt.subplot(2,1,1)
plt.plot(x, label='TMIN238')
plt.plot(y, '.-k', label='TMAX238')
plt.legend(); plt.show()

plt.figure(2, figsize=(15, 5))
plt.plot(x, y, '.', label=f'Datos de la muestra')
plt.xlabel('TMIN238'); plt.ylabel('TMAX238')
plt.grid(); plt.legend(); plt.show()

print('Existe una fuerte relación')

### 3.2 Planteamiento del problema: la distribución de los residuos es la clave

In [ ]:

xs = np.linspace(np.min(x)-2, np.max(x)+2, 1000)
yest_recta1 = 9 + 1.22 * xs
yest_recta2 = 3*9 + 1.22 * xs
yest_recta3 = -15.8222 + (3 * 1.22) * xs    #  21.4 - 3 * 1.22 * 10.17 + (3 * 1.22) * xs

plt.figure(1, figsize=(15, 5))
plt.plot(x, y, '.', alpha = 0.2, label=f'Datos de la muestra')
plt.plot(xs, yest_recta1, '-r', linewidth = 4, label='Recta 1')
plt.plot(xs, yest_recta2, '-b', linewidth = 4, label='Recta 2')
plt.plot(xs, yest_recta3, '-g', linewidth = 4, label='Recta 3')
plt.plot(np.mean(x), np.mean(y), 'sk', markersize = 10, label = '(x_mean, y_mean)')
plt.grid()
plt.xlabel('x'); plt.ylabel('y')
plt.legend()
plt.show()


# residuos de las tres rectas
err1 =  y - (9 + 1.22 * x)
err2 =  y - (3*9 + 1.22 * x)
err3 =  y - (-15.8222 + (3 * 1.22) * x)


# representación de la distribución supuesta y la observada
xs_err = np.linspace(-60, 60, 1000)

plt.figure(2, figsize=(15, 5))

ax1 = plt.subplot(1,3,1)
plt.hist(err1, 40, density=True, alpha = 0.6, label=f'Histograma errores recta 1')
vaErr = stats.norm(loc = np.mean(err1), scale = np.std(err1))
plt.plot(xs_err, vaErr.pdf(xs_err), '-r', linewidth= 3, label=f'Normal ajustada')
plt.vlines(0, 0, np.max(vaErr.pdf(xs_err)), color = 'black')
plt.xlabel('error recta 1'); plt.grid()

plt.subplot(1,3,2, sharex = ax1)
plt.hist(err2, 40, density=True, alpha = 0.6, label=f'Histograma errores recta 2')
vaErr = stats.norm(loc = np.mean(err2), scale = np.std(err2))
plt.plot(xs_err, vaErr.pdf(xs_err), '-b', linewidth= 3, label=f'Normal ajustada')
plt.vlines(0, 0, np.max(vaErr.pdf(xs_err)), color = 'black')
plt.xlabel('error recta 2'); plt.grid()

plt.subplot(1,3,3, sharex = ax1)
plt.hist(err3, 40, density=True, alpha = 0.6, label=f'Histograma errores recta 3')
vaErr = stats.norm(loc = np.mean(err3), scale = np.std(err3))
plt.plot(xs_err, vaErr.pdf(xs_err), '-g', linewidth= 3, label=f'Normal ajustada')
plt.vlines(0, 0, np.max(vaErr.pdf(xs_err)), color = 'black')
plt.xlabel('error recta 3'); plt.grid()
plt.show()


### 3.3 Construir un estimador para TMAX238 a partir de TMIN238

In [ ]:
# definimos la función de verosimilitud logarítmica para maximizarla
    
def verosimilitud_log(parameters):
    # sacamos los parámetros
    beta0, beta1, beta2, std_dev = parameters
    
    # estimamos la salida a partir de esos parámetros
    pred = beta0 + beta1*x
    pred = beta0 + beta1*x + beta2 * x**2
    # calculamos la diferencia entre la salida deseada y la estimada (el error)
    err = y - pred
    
    # La función de verosimilitud logarítmica considerando que el error se
    # distribuye como una normal de media cero y varianza cte (a determinar)
    logLikelihood = np.sum(stats.norm.logpdf(err, loc=0, scale=std_dev))  # <--- logpdf es el log de la pdf, ya está preparado ...
    
    # Como la función minimiza y queremos los params que maximizan la verosimilitud
    # simplemente la multimplicamos por -1: max(loglike) = min(-loglike)
    neg_logLikelihood = -1 * logLikelihood
    return neg_logLikelihood 

In [ ]:
# Maximizamos la función de verosimilitud logarítmica (minimizando -1 * funcion)
mle_model = opt.minimize(verosimilitud_log, np.array([1,1,1,1]), method='L-BFGS-B')

# cogemos los coefs estimados
beta0 = mle_model.x[0]
beta1 = mle_model.x[1]
beta2 = mle_model.x[2]
std_err = mle_model.x[3]

print(mle_model)
print(f'Ecuación estimada por MV: y = {beta0} + {beta1} * x + {beta2} * x^2 + err,')
print(f'en donde el error err se distribuye como una normal de media 0 y std = {std_err}')

In [ ]:
# representamos el estimador, su ecuación
xs = np.linspace(np.min(x)-2, np.max(x)+2, 1000)
yest_xs = beta0 + beta1 * xs + beta2 * xs**2

plt.figure(1, figsize=(15, 5))
plt.plot(x, y, '.', label=f'Datos de la muestra')
plt.plot(xs, yest_xs, '-r', label=f'Polinomio y = {beta0} + {beta1} * x + {beta2} * x^2')
plt.xlabel('x'); plt.ylabel('y')
plt.legend()
plt.show()

### 3.4 Estimar puntualmente TMAX238 a partir del estimador creado, basado en TMIN238

In [ ]:
# usamos el valor esperado del estimador (dado por la cuadrática)
yest = beta0 + beta1 * x + beta2 * x**2

# errores cometidos
err = y - yest

plt.figure(1, figsize=(20, 7))
ax1=plt.subplot(2,1,1)
plt.plot(x, label='x: TMIN238')
plt.plot(y, '.-k', label='y: TMAX238')
plt.plot(yest, '.-r', label='yest')
plt.title ('Primeros 2 años (2x365 días)')
plt.legend()
plt.subplot(2,1,2,sharex=ax1)
plt.stem(err)
plt.title('Error (diferencia real - estimado)')
plt.xlim([1, 2*365])
plt.show()

### 3.5 Verificar que la distribución de los errores es realmente una normal

In [ ]:
# v.a. que representa la distribución del error según nuestra hipótesis
vaErr = stats.norm(loc = 0, scale = std_err)

# representación de la distribución supuesta y la observada
xs = np.linspace(np.min(err)-2, np.max(err)+2, 1000)

plt.figure(2, figsize=(15, 5))
plt.hist(err, 40, density=True, label=f'Histograma errores')
plt.plot(xs, vaErr.pdf(xs), '-r', linewidth= 3, label=f'Hipótesis: err~N(0,{std_err})')
plt.xlabel('err'); plt.legend(); plt.show()

### 3.6 Crear un intervalo de confianza al 90% para cada día a partir de estimador

In [ ]:
# Intervalos de confianza a partir del error estandar del estimador creado
gama = 0.90
alfa = 1 - gama
lim_inf = vaErr.ppf(alfa/2)
lim_sup = vaErr.ppf(1-alfa/2)
print(f'Limites del IC del error (simétrico respecto al cero): [{lim_inf},{lim_sup}]')

# creamos un IC para cada día centrando el intervalo en la estimación puntual de cada día
lim_sup_banda = yest + lim_inf
lim_inf_banda = yest + lim_sup

# marcamos los días que están fuera del intervalo
ind_dentro = (lim_inf_banda-y>=0) & (y-lim_sup_banda>=0)
ind_fuera  = (lim_inf_banda-y<0) | (y-lim_sup_banda<0)

# Confirmamos si el IC tiene la exactitud que debería según el nivel de confianza utilizado
num_dias_dentro_IC = np.sum(ind_dentro)
num_dias_fuera_IC  = np.sum(ind_fuera)
prctje = 100 * num_dias_fuera_IC/n
print(f'Número de días dentro del intervalo: {num_dias_dentro_IC}, fuera: {num_dias_fuera_IC}. ' 
      f'Hay un {prctje} % de días fuera de su IC.')

In [ ]:
# graficamente 
plt.figure(1, figsize=(20, 5))
plt.plot(y, '.-k', label='y real')
plt.plot(yest, '.-r', label='y estimada (centro del IC)')
plt.plot(y[ind_fuera], 'om', label='Fuera del IC')
plt.fill_between(range(0,n), lim_sup_banda, lim_inf_banda, color='green', alpha=.6, label='IC estimado')
plt.xlim([1, 100]) # 100 puntos
plt.title('Primeros 100 días')
plt.legend()
plt.show()

plt.figure(2, figsize=(20, 5))
plt.plot(y, '.-k', label='y real')
plt.plot(y[ind_fuera], 'om', label='Fuera del IC')
plt.fill_between(range(0,n), lim_sup_banda, lim_inf_banda, color='green', alpha=.6, label='IC estimado')
plt.xlim([1, 365]) # 100 puntos
plt.title('Primer año (365 días)')
plt.legend()
plt.show()

In [ ]:
# Curiosidad: El modelo anterior es un modelo de regresión lineal múltiple ajustado por mínimos cuadrados ordinarios (OLS)
# compara los coefs con los obtenidos (son los mismos)
from statsmodels import api

vars_entrada = np.array(np.transpose([x, x**2]))

features = api.add_constant(vars_entrada)
model = api.OLS(y, features).fit()
model.summary()